# License Plate Detection & OCR

This notebook combines YOLOv8 for license plate detection and EasyOCR for text extraction.

In [ ]:
# Cross-Platform Environment Setup
import os
import sys

# Detect environment
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

IN_KAGGLE = 'KAGGLE_URL_BASE' in os.environ

if IN_COLAB or IN_KAGGLE:
    print('Running in cloud environment. Installing dependencies...')
    !pip install -qU ultralytics wandb roboflow python-dotenv supervision easyocr cvzone
    
    if IN_COLAB:
        from google.colab import userdata
        os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY') or ''
        os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY') or ''
    elif IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        os.environ['WANDB_API_KEY'] = user_secrets.get_secret('WANDB_API_KEY') or ''
        os.environ['ROBOFLOW_API_KEY'] = user_secrets.get_secret('ROBOFLOW_API_KEY') or ''
else:
    print('Running locally. Loading from .env...')
    try:
        from dotenv import load_dotenv
        load_dotenv('../.env')
    except ImportError:
        print('python-dotenv not installed.')

In [ ]:
import numpy as np
import cv2
from ultralytics import YOLO
import cvzone
import easyocr
from IPython.display import display, clear_output, Image as IPImage
import ipywidgets as widgets

# ── Global Config ──────────────────
VIDEO_PATH      = "../assets/Video Test/ocr-trying/nr.mp4"
IMAGE_PATH      = "../assets/ocr-trying/10google2-jumbo.png"
MODEL_PATH      = "../models/ocr/pytorch/v4/1/license-plate-ocr-v4.pt"
CONF_THRESHOLD  = 0.1
OCR_CONF_MIN    = 0.1
LANG            = 'en'
PLATE_CLASS     = "License_Plate"
# ────────────────────────────────

# Initialize models
model = YOLO(MODEL_PATH)
reader = easyocr.Reader([LANG], gpu=True)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
def preprocess_for_ocr(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape
    if h < 80:
        scale = 80 / h
        gray = cv2.resize(gray, (int(w * scale), 80), interpolation=cv2.INTER_CUBIC)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(4, 4))
    gray = clahe.apply(gray)
    return cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

def perform_ocr(image_array):
    if image_array is None or image_array.size == 0: return ""
    results = reader.readtext(image_array)
    filtered = [text for (_, text, conf) in results if conf >= OCR_CONF_MIN]
    return " ".join(filtered).strip()

def show_frame(frame):
    if IN_COLAB or IN_KAGGLE:
        _, encoded_img = cv2.imencode('.jpg', frame)
        display(widgets.Image(value=encoded_img.tobytes()))
        clear_output(wait=True)
    else:
        cv2.imshow("OCR Detection", frame)

## Run OCR on Video

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
while cap.isOpened():
    ret, frame = cap.read()
    if not ret: break
    
    results = model.predict(frame, conf=CONF_THRESHOLD, verbose=False)
    for box in results[0].boxes:
        if model.names[int(box.cls[0])] != PLATE_CLASS: continue
        
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        crop = frame[y1:y2, x1:x2]
        if crop.size > 0:
            text = perform_ocr(preprocess_for_ocr(crop))
            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cvzone.putTextRect(frame, text, (x1, y1-10), scale=1, thickness=1)
            
    show_frame(frame)
    if not (IN_COLAB or IN_KAGGLE):
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
if not (IN_COLAB or IN_KAGGLE):
    cv2.destroyAllWindows()